In [2]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

In [3]:
fire = pd.read_csv("fire_mah.csv")
print("Rows:", len(fire))
print("Columns:", len(fire.columns))
print(fire.head())

Rows: 36078
Columns: 13
   latitude  longitude  bright_ti4  scan  track    acq_date acq_time  \
0  20.17263   79.91102      330.73  0.35   0.56  2025-01-01    07:16   
1  19.38578   75.51940      328.87  0.69   0.74  2025-01-01    07:16   
2  19.97110   76.39579      334.76  0.59   0.70  2025-01-01    07:16   
3  19.96714   76.39572      341.69  0.59   0.70  2025-01-01    07:16   
4  20.88234   79.15154      330.27  0.37   0.58  2025-01-01    07:16   

  satellite confidence version  bright_ti5    frp daynight  
0       N21    nominal  2.0NRT      296.20   2.80        D  
1       N21    nominal  2.0NRT      297.60   4.76        D  
2       N21    nominal  2.0NRT      301.02  13.62        D  
3       N21    nominal  2.0NRT      302.18   8.28        D  
4       N21    nominal  2.0NRT      300.82   2.35        D  


In [6]:
fire["Date"] = pd.to_datetime(
    fire["acq_date"],
    errors="coerce"
).dt.date

print("Missing Dates:")
print(fire["Date"].isna().sum())

Missing Dates:
0


In [7]:
# =====================================================
# LOAD DISTRICT SHAPEFILE
# =====================================================

districts = gpd.read_file("gadm41_IND_2.shp")

mh = districts[
    districts["NAME_1"] == "Maharashtra"
].copy()

print("\nDistricts Found:")
print(mh["NAME_2"].nunique())



Districts Found:
36


In [8]:
# =====================================================
# CREATE GEOMETRY
# =====================================================

fire_gdf = gpd.GeoDataFrame(
    fire,
    geometry=gpd.points_from_xy(
        fire["longitude"],
        fire["latitude"]
    ),
    crs="EPSG:4326"
)


In [9]:
import pandas as pd

print(pd.read_csv("fire_mah.csv", nrows=5).columns.tolist())
print(pd.read_csv("Mah_Daily_AirPollution_2025.csv", nrows=5).columns.tolist())
print(pd.read_csv("ERA_2025_clean.csv", nrows=5).columns.tolist())

['latitude', 'longitude', 'bright_ti4', 'scan', 'track', 'acq_date', 'acq_time', 'satellite', 'confidence', 'version', 'bright_ti5', 'frp', 'daynight']
['Date', 'District', 'HCHO', 'CO', 'NO2', 'SO2', 'O3']
['district', 'date', 'u10', 'v10', 't2m', 'sp', 'blh']


In [10]:
# =====================================================
# DISTRICT MAPPING
# =====================================================

fire_gdf = gpd.sjoin(
    fire_gdf,
    mh[["NAME_2", "geometry"]],
    how="left",
    predicate="within"
)

fire_gdf.rename(
    columns={"NAME_2": "District"},
    inplace=True
)

print("\nDistrict Mapping Complete")

print("Unique Districts:")
print(fire_gdf["District"].nunique())

print("\nTop District Counts")

print(
    fire_gdf["District"]
    .value_counts()
    .head(20)
)

print("\nMissing Districts")

print(
    fire_gdf["District"]
    .isna()
    .sum()
)


District Mapping Complete
Unique Districts:
36

Top District Counts
District
Raigarh        1866
Pune           1401
Solapur         968
Satara          949
Ahmadnagar      795
Thane           737
Ratnagiri       709
Chandrapur      670
Garhchiroli     634
Nagpur          577
Sindhudurg      498
Amravati        493
Kolhapur        491
Yavatmal        457
Nanded          436
Bid             432
Nashik          411
Sangli          402
Osmanabad       351
Aurangabad      340
Name: count, dtype: int64

Missing Districts
19764


In [11]:
# =====================================================
# KEEP REQUIRED COLUMNS
# =====================================================

clean_fire = fire_gdf[
    ["Date", "District", "frp"]
].copy()

print("\nAfter Column Removal")

print(clean_fire.head())

print("Shape:", clean_fire.shape)


After Column Removal
         Date    District    frp
0  2025-01-01  Chandrapur   2.80
1  2025-01-01         Bid   4.76
2  2025-01-01     Buldana  13.62
3  2025-01-01     Buldana   8.28
4  2025-01-01      Nagpur   2.35
Shape: (36078, 3)


In [12]:
# =====================================================
# SAVE
# =====================================================

clean_fire.to_csv(
    "fire_mah_cleaned.csv",
    index=False
)

print("\nSaved: fire_mah_cleaned.csv")


Saved: fire_mah_cleaned.csv
